# Results Summary Dashboard

Loads CSV outputs and presents a compact comparison dashboard.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.plot_style import apply_plot_style

apply_plot_style()
sns.set_theme(style="whitegrid")
output_dir = Path("outputs")
csv_files = sorted(output_dir.glob("*.csv"))
summary = pd.DataFrame({"file": [p.name for p in csv_files], "rows": [pd.read_csv(p).shape[0] for p in csv_files], "cols": [pd.read_csv(p).shape[1] for p in csv_files]})
summary

## Available Output Tables

In [ ]:
display(summary)

## Quick Views

In [ ]:
tables = {p.stem: pd.read_csv(p) for p in csv_files}
for name, frame in tables.items():
    print(f"
=== {name} ===")
    display(frame.head(10))

## Segment Performance Snapshot

In [ ]:
if "segment_performance" in tables:
    seg = tables["segment_performance"]
    pivot = seg.pivot_table(index=["segment_type", "segment_value"], columns="model", values="roc_auc")
    display(pivot.head(20))
    plt.figure(figsize=(14, 8))
    sns.heatmap(pivot, cmap="RdYlGn", vmin=0.5, vmax=1.0)
    plt.title("Per-Segment ROC AUC by Model")
    plt.tight_layout()

## Calibration and SHAP Artifacts

In [ ]:
for expected in ["calibration_brier_comparison.csv", "shap_feature_importance.csv", "ablation_summary.csv"]:
    path = output_dir / expected
    if path.exists():
        print(f"Found: {expected}")
        display(pd.read_csv(path).head(10))
    else:
        print(f"Missing: {expected}")